# Fed 3M Forward Audit — Bloomberg `USSOC Curncy`

This notebook reproduces the monthly Bloomberg series used to audit the thesis variable `fed_3m_forward`.

**Bloomberg source**
- Ticker: `USSOC Curncy`
- Field: `PX_LAST`
- Instrument: 3-month USD Overnight Indexed Swap (OIS) fixed rate
- Frequency convention: calendar month-end with previous available quote carried to month-end (`fill='PREV'`)
- Audit window: January 2010 through December 2025

The output is intended for the thesis Data Freeze Gate and can be compared against the monthly `fed_3m_forward` field in `panel_main_and_robustness.xlsx`.

In [ ]:
import bql
import pandas as pd

from pathlib import Path
from urllib.parse import quote
from IPython.display import HTML, display


## Download helper

In [ ]:
def download_file(file_path, obj=None, index=False):
    """Save a file in the BQuant/Jupyter project and create a browser download button."""

    path = Path(file_path)
    if not path.is_absolute():
        path = Path.cwd() / path

    path.parent.mkdir(parents=True, exist_ok=True)

    if obj is not None:
        ext = path.suffix.lower()

        if isinstance(obj, (pd.DataFrame, pd.Series)):
            if ext == ".xlsx":
                obj.to_excel(path, index=index)
            elif ext == ".csv":
                obj.to_csv(path, index=index)
            else:
                raise ValueError("For pandas objects use .xlsx or .csv")
        elif isinstance(obj, str):
            path.write_text(obj, encoding="utf-8")
        elif isinstance(obj, bytes):
            path.write_bytes(obj)
        else:
            raise TypeError(f"Unsupported object type: {type(obj)}")

    if not path.exists():
        raise FileNotFoundError(path)

    cwd = Path.cwd()
    relative_path = path.relative_to(cwd)
    encoded_path = "/".join(quote(part) for part in relative_path.parts)
    filename = path.name

    html = f"""
    <button
        style="padding: 8px 16px; font-size: 14px; cursor: pointer;"
        onclick="
            const currentPath = window.location.pathname;
            const basePath = currentPath.split('/lab/')[0];
            const downloadUrl = window.location.origin + basePath + '/files/{encoded_path}?download=1';
            const a = document.createElement('a');
            a.href = downloadUrl;
            a.download = '{filename}';
            document.body.appendChild(a);
            a.click();
            document.body.removeChild(a);
        "
    >
        ⬇ Download {filename}
    </button>
    """

    print("Saved:")
    print(path)
    display(HTML(html))
    return path


## Bloomberg setup

In [ ]:
bq = bql.Service()

TICKER = "USSOC Curncy"
START_DATE = "2010-01-31"
END_DATE = "2025-12-31"


## Monthly extraction

`frq='CM'` requests calendar month-end observations. `fill='PREV'` carries the most recent available market quote to calendar month-end when the month ends on a non-quotation day.

In [ ]:
item = bq.data.px_last(
    fill="PREV",
    dates=bq.func.range(
        start=START_DATE,
        end=END_DATE,
        frq="CM",
        frqstep=1,
    ),
)

req = bql.Request(TICKER, {"PX_LAST": item})
res = bq.execute(req)
df = res[0].df().reset_index()

df = df.rename(
    columns={
        "DATE": "calendar_month_end",
        "PX_LAST": "ussoc_px_last",
    }
)

df["calendar_month_end"] = pd.to_datetime(
    df["calendar_month_end"],
    errors="coerce",
)

display(df)


## Diagnostics

In [ ]:
print("=" * 70)
print("USSOC MONTH-END SUMMARY")
print("=" * 70)
print("Rows:", len(df))
print("Missing PX_LAST:", df["ussoc_px_last"].isna().sum())
print("\nFirst 5:")
print(df.head().to_string(index=False))
print("\nLast 5:")
print(df.tail().to_string(index=False))


## Export reproducible monthly series

In [ ]:
output_path = (
    "fed_3m_forward_audit/"
    "fed_3m_forward_ussoc_monthly_2010_2025.csv"
)

download_file(
    output_path,
    df,
    index=False,
)


## Audit interpretation

The extracted series is the Bloomberg-side benchmark for the Data Freeze Gate panel-match test. The econometric panel variable is named `fed_3m_forward`, but the Bloomberg instrument identified in the audit is the **spot 3-month USD OIS fixed rate**. It is forward-looking in the economic sense because it reflects market expectations for overnight rates over the next three months; it is not a forward-starting 3M×3M OIS.